In [25]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

## PreRequisites

In [26]:
import pandas as pd
import pickle

from radp.digital_twin.utils.gis_tools import GISTools
from notebooks.radp_library import calculate_received_power, calc_log_distance, calc_relative_bearing
from notebooks.radp_library import get_percell_data

In [27]:
with open('E:/Repositories/maveric/notebooks/data/sim_data/processed_training_data.pkl', 'rb') as f:
    loaded_dict = pickle.load(f)

print(loaded_dict)

{'cell_1':       mock_ue_id   longitude   latitude  tick  cell_lat  cell_lon cell_id  \
0              5  -43.264737 -30.862211    20     -90.0    -180.0  cell_1   
1             10   54.978193 -55.259842    59     -90.0    -180.0  cell_1   
2             12  -98.676642  40.779019    56     -90.0    -180.0  cell_1   
3             11    5.606533  12.413489    36     -90.0    -180.0  cell_1   
4             14 -134.815655 -58.151045    87     -90.0    -180.0  cell_1   
...          ...         ...        ...   ...       ...       ...     ...   
1995          15   86.855787  41.319434    41     -90.0    -180.0  cell_1   
1996          16  -84.432536 -38.512045    60     -90.0    -180.0  cell_1   
1997          13  142.954432 -16.862212    82     -90.0    -180.0  cell_1   
1998          19   89.254786 -82.596233    27     -90.0    -180.0  cell_1   
1999           4   20.088340 -55.786882    34     -90.0    -180.0  cell_1   

      cell_az_deg  cell_carrier_freq_mhz  log_distance  cell_rxp

In [28]:
simple_ue = pd.read_csv('E:/Repositories/maveric/notebooks/data/sim_data/UE_data_20UE_100ticks.csv')
topology = pd.read_csv('E:/Repositories/maveric/notebooks/data/sim_data/topology.csv')

In [29]:
topology

,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,35.690556,139.691944,cell_1,0,2100
1,35.690556,139.691944,cell_2,120,2100
2,35.690556,139.691944,cell_3,240,2100


In [30]:
simple_ue.drop(columns=['mock_ue_id', 'tick'], inplace=True)
simple_ue

,longitude,latitude
0,-22.625309,59.806764
1,119.764151,54.857584
2,72.095437,-20.253892
3,-67.548009,-38.100941
4,59.867089,-83.103930
...,...,...
1995,45.564260,42.609846
1996,132.457280,17.241235
1997,-101.217659,72.295988
1998,-16.480045,-26.656397


## Functions

In [31]:
def f0(data,topology):
    if topology["cell_id"].dtype == object:
            topology["cell_id"] = (
                topology["cell_id"].str.replace("cell_", "").astype(int)
            )
    data["key"] = 1
    topology["key"] = 1
    combined_df = pd.merge(data, topology, on="key").drop("key", axis=1)
    return combined_df

In [32]:
def f1(cartesian_df):
    cartesian_df["log_distance"] = cartesian_df.apply(
        lambda row: GISTools.get_log_distance(
            row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
        ),
        axis=1,
    )
    return cartesian_df

In [33]:
def f2(cartesian_df):
    cartesian_df["cell_rxpwr_dbm"] = cartesian_df.apply(
        lambda row: calculate_received_power(
            row["log_distance"], row["cell_carrier_freq_mhz"]
        ),
        axis=1,
    )
    return cartesian_df

In [34]:
def f3(cartesian_df):
    cartesian_df["relative_bearing"] = cartesian_df.apply(
        lambda row: GISTools.get_relative_bearing(
            row["cell_az_deg"],
            row["cell_lat"],
            row["cell_lon"],
            row["latitude"],
            row["longitude"],
        ),
        axis=1,
    )
    return cartesian_df

In [35]:
def preprocess_ue_data(data, topology):
    cartesian_df = f0(data, topology)
    cartesian_df = f1(cartesian_df)
    return f2(cartesian_df)


In [36]:
def prepare_train_or_update_data(df):
    update_data = calc_log_distance(df)
    update_data = calc_relative_bearing(update_data)
    update_data.drop(columns=['longitude', 'latitude','cell_lat','cell_lon', 'cell_az_deg','cell_carrier_freq_mhz'], inplace=True)

    train_per_cell_df = [x for _, x in update_data.groupby("cell_id")]
    n_cell = len(topology.index)

    metadata_df = pd.DataFrame(
        {
            "cell_id": [cell_id for cell_id in topology.cell_id],
            "idx": [i + 1 for i in range(n_cell)],
        }
    )
    idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))
    n_samples_train = []
    for df in train_per_cell_df:
        n_samples_train.append(df.shape[0])

    train_per_cell_df_processed = []
    for i in range(n_cell):
        train_per_cell_df_processed.append(
            get_percell_data(
                data_in=train_per_cell_df[i],
                choose_strongest_samples_percell=False,
                n_samples=n_samples_train[i],
            )[0][0]
        )

    training_data = {}

    for i, df in enumerate(train_per_cell_df_processed):
        train_cell_id = idx_cell_id_mapping[i + 1]
        training_data[train_cell_id] = df
    
    return training_data

## Bebugging

In [37]:
cartesian_df = f0(simple_ue, topology)
cartesian_df

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,-22.625309,59.806764,35.690556,139.691944,1,0,2100
1,-22.625309,59.806764,35.690556,139.691944,2,120,2100
2,-22.625309,59.806764,35.690556,139.691944,3,240,2100
3,119.764151,54.857584,35.690556,139.691944,1,0,2100
4,119.764151,54.857584,35.690556,139.691944,2,120,2100
...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,35.690556,139.691944,2,120,2100
5996,-16.480045,-26.656397,35.690556,139.691944,3,240,2100
5997,34.222834,63.970820,35.690556,139.691944,1,0,2100
5998,34.222834,63.970820,35.690556,139.691944,2,120,2100


In [38]:
cartesian_df = f1(cartesian_df)
cartesian_df

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance
0,-22.625309,59.806764,35.690556,139.691944,1,0,2100,16.043691
1,-22.625309,59.806764,35.690556,139.691944,2,120,2100,16.043691
2,-22.625309,59.806764,35.690556,139.691944,3,240,2100,16.043691
3,119.764151,54.857584,35.690556,139.691944,1,0,2100,14.780124
4,119.764151,54.857584,35.690556,139.691944,2,120,2100,14.780124
...,...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,35.690556,139.691944,2,120,2100,16.681342
5996,-16.480045,-26.656397,35.690556,139.691944,3,240,2100,16.681342
5997,34.222834,63.970820,35.690556,139.691944,1,0,2100,15.788136
5998,34.222834,63.970820,35.690556,139.691944,2,120,2100,15.788136


In [39]:
cartesian_df = f2(cartesian_df)
cartesian_df

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance,cell_rxpwr_dbm
0,-22.625309,59.806764,35.690556,139.691944,1,0,2100,16.043691,-100.000472
1,-22.625309,59.806764,35.690556,139.691944,2,120,2100,16.043691,-100.000472
2,-22.625309,59.806764,35.690556,139.691944,3,240,2100,16.043691,-100.000472
3,119.764151,54.857584,35.690556,139.691944,1,0,2100,14.780124,-99.287948
4,119.764151,54.857584,35.690556,139.691944,2,120,2100,14.780124,-99.287948
...,...,...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,35.690556,139.691944,2,120,2100,16.681342,-100.339006
5996,-16.480045,-26.656397,35.690556,139.691944,3,240,2100,16.681342,-100.339006
5997,34.222834,63.970820,35.690556,139.691944,1,0,2100,15.788136,-99.861003
5998,34.222834,63.970820,35.690556,139.691944,2,120,2100,15.788136,-99.861003


In [40]:
cartesian_df = f3(cartesian_df)
cartesian_df

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance,cell_rxpwr_dbm,relative_bearing
0,-22.625309,59.806764,35.690556,139.691944,1,0,2100,16.043691,-100.000472,351.153872
1,-22.625309,59.806764,35.690556,139.691944,2,120,2100,16.043691,-100.000472,231.153872
2,-22.625309,59.806764,35.690556,139.691944,3,240,2100,16.043691,-100.000472,111.153872
3,119.764151,54.857584,35.690556,139.691944,1,0,2100,14.780124,-99.287948,330.617727
4,119.764151,54.857584,35.690556,139.691944,2,120,2100,14.780124,-99.287948,210.617727
...,...,...,...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,35.690556,139.691944,2,120,2100,16.681342,-100.339006,167.318054
5996,-16.480045,-26.656397,35.690556,139.691944,3,240,2100,16.681342,-100.339006,47.318054
5997,34.222834,63.970820,35.690556,139.691944,1,0,2100,15.788136,-99.861003,332.079388
5998,34.222834,63.970820,35.690556,139.691944,2,120,2100,15.788136,-99.861003,212.079388


## Rewriting Update (Training From Scratch)

In [41]:
df = preprocess_ue_data(simple_ue, topology)
df1 = df.copy()

In [42]:
bayesian_digital_twins = {}

In [43]:
try:
    if not isinstance(df, pd.DataFrame):
        raise TypeError("The input 'new_data' must be a pandas DataFrame.")

    expected_columns = {"longitude", "latitude", "cell_lat", "cell_lon", "cell_id", "cell_az_deg", "cell_carrier_freq_mhz", "cell_rxpwr_dbm"}
    if not expected_columns.issubset(df.columns):
        expected_columns = {"longitude", "latitude"}
        if not expected_columns.issubset(df.columns):
            raise ValueError(
                f"The input DataFrame must contain the following columns: {expected_columns}"
        )
        else:
            df = preprocess_ue_data(simple_ue, topology)
            
    training_data = prepare_train_or_update_data(df)
    print(training_data)
    if bayesian_digital_twins:
        # TODO: Update BDT
        pass
    else:
        # TODO: Create BDT from scratch
        pass

        # TODO: Add Train

except TypeError as te:
    print(f"TypeError: {te}")
except ValueError as ve:
    print(f"ValueError: {ve}")
except KeyError as ke:
    print(f"KeyError: {ke}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

{1:       cell_id  log_distance  cell_rxpwr_dbm  relative_bearing
0           1     16.782517     -100.391528         27.934319
1           1     16.372865     -100.176879        219.107651
2           1     16.081985     -100.021179         40.229093
3           1     16.367184     -100.173865        309.143895
4           1     16.386669     -100.184200        143.622155
...       ...           ...             ...               ...
1995        1     15.335904      -99.608574        294.404550
1996        1     16.597103     -100.295032        108.097728
1997        1     15.583699      -99.747797        176.071117
1998        1     16.414202     -100.198782        186.640616
1999        1     16.525990     -100.257736        223.811745

[2000 rows x 4 columns], 2:       cell_id  log_distance  cell_rxpwr_dbm  relative_bearing
0           2     16.782517     -100.391528        267.934319
1           2     16.372865     -100.176879         99.107651
2           2     16.081985     -100.

### Testing

In [44]:
update_data = calc_log_distance(df1)
update_data = calc_relative_bearing(update_data)

update_data.drop(columns=['longitude', 'latitude','cell_lat','cell_lon', 'cell_az_deg','cell_carrier_freq_mhz'], inplace=True)

In [45]:
update_data

,cell_id,log_distance,cell_rxpwr_dbm,relative_bearing
0,1,16.043691,-100.000472,351.153872
1,2,16.043691,-100.000472,231.153872
2,3,16.043691,-100.000472,111.153872
3,1,14.780124,-99.287948,330.617727
4,2,14.780124,-99.287948,210.617727
...,...,...,...,...
5995,2,16.681342,-100.339006,167.318054
5996,3,16.681342,-100.339006,47.318054
5997,1,15.788136,-99.861003,332.079388
5998,2,15.788136,-99.861003,212.079388


In [46]:
train_per_cell_df = [x for _, x in update_data.groupby("cell_id")]
n_cell = len(topology.index)

metadata_df = pd.DataFrame(
    {
        "cell_id": [cell_id for cell_id in topology.cell_id],
        "idx": [i + 1 for i in range(n_cell)],
    }
)
idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))
desired_idxs = [1 + r for r in range(n_cell)]

n_samples_train = []
for df in train_per_cell_df:
    n_samples_train.append(df.shape[0])

train_per_cell_df_processed = []
for i in range(n_cell):
    train_per_cell_df_processed.append(
        get_percell_data(
            data_in=train_per_cell_df[i],
            choose_strongest_samples_percell=False,
            n_samples=n_samples_train[i],
        )[0][0]
    )

training_data = {}

for i, df in enumerate(train_per_cell_df_processed):
    train_cell_id = idx_cell_id_mapping[i + 1]
    training_data[train_cell_id] = df

In [47]:
training_data

{1:       cell_id  log_distance  cell_rxpwr_dbm  relative_bearing
 0           1     16.782517     -100.391528         27.934319
 1           1     16.372865     -100.176879        219.107651
 2           1     16.081985     -100.021179         40.229093
 3           1     16.367184     -100.173865        309.143895
 4           1     16.386669     -100.184200        143.622155
 ...       ...           ...             ...               ...
 1995        1     15.335904      -99.608574        294.404550
 1996        1     16.597103     -100.295032        108.097728
 1997        1     15.583699      -99.747797        176.071117
 1998        1     16.414202     -100.198782        186.640616
 1999        1     16.525990     -100.257736        223.811745
 
 [2000 rows x 4 columns],
 2:       cell_id  log_distance  cell_rxpwr_dbm  relative_bearing
 0           2     16.782517     -100.391528        267.934319
 1           2     16.372865     -100.176879         99.107651
 2           2     16

In [48]:
for train_cell_id, training_data_idx in training_data.items():
    print(f"Cell ID: {train_cell_id}")
    print(f"Training Data Index: {training_data_idx}")

Cell ID: 1
Training Data Index:       cell_id  log_distance  cell_rxpwr_dbm  relative_bearing
0           1     16.782517     -100.391528         27.934319
1           1     16.372865     -100.176879        219.107651
2           1     16.081985     -100.021179         40.229093
3           1     16.367184     -100.173865        309.143895
4           1     16.386669     -100.184200        143.622155
...       ...           ...             ...               ...
1995        1     15.335904      -99.608574        294.404550
1996        1     16.597103     -100.295032        108.097728
1997        1     15.583699      -99.747797        176.071117
1998        1     16.414202     -100.198782        186.640616
1999        1     16.525990     -100.257736        223.811745

[2000 rows x 4 columns]
Cell ID: 2
Training Data Index:       cell_id  log_distance  cell_rxpwr_dbm  relative_bearing
0           2     16.782517     -100.391528        267.934319
1           2     16.372865     -100.176879

In [49]:
print(loaded_dict)

{'cell_1':       mock_ue_id   longitude   latitude  tick  cell_lat  cell_lon cell_id  \
0              5  -43.264737 -30.862211    20     -90.0    -180.0  cell_1   
1             10   54.978193 -55.259842    59     -90.0    -180.0  cell_1   
2             12  -98.676642  40.779019    56     -90.0    -180.0  cell_1   
3             11    5.606533  12.413489    36     -90.0    -180.0  cell_1   
4             14 -134.815655 -58.151045    87     -90.0    -180.0  cell_1   
...          ...         ...        ...   ...       ...       ...     ...   
1995          15   86.855787  41.319434    41     -90.0    -180.0  cell_1   
1996          16  -84.432536 -38.512045    60     -90.0    -180.0  cell_1   
1997          13  142.954432 -16.862212    82     -90.0    -180.0  cell_1   
1998          19   89.254786 -82.596233    27     -90.0    -180.0  cell_1   
1999           4   20.088340 -55.786882    34     -90.0    -180.0  cell_1   

      cell_az_deg  cell_carrier_freq_mhz  log_distance  cell_rxp